In [1]:
import os
dir_path = os.getcwd()
print("The directory of this script is:", dir_path)
root_path = os.path.dirname(dir_path)
print("The root directory is:", root_path)

The directory of this script is: c:\Users\HP\Desktop\Projects\NodeRAG\retrieval
The root directory is: c:\Users\HP\Desktop\Projects\NodeRAG


In [2]:
import sys
sys.path.append(root_path)
from graphs.Node import Node

In [3]:
import pickle
import numpy as np
with open(f"{root_path}/graphs/data/graphs/G6_entity_and_chunk_resolution_graph.pkl", "rb") as f:
    medical_g6 = pickle.load(f)
medical_g6

{'medical-0-S-0': <graphs.Node.Node at 0x1fa8bfdfd00>,
 'medical-0-S-0-R-0': <graphs.Node.Node at 0x1faa339d8d0>,
 'medical-0-S-0-R-1': <graphs.Node.Node at 0x1faa339d900>,
 'medical-0-S-0-R-2': <graphs.Node.Node at 0x1faa339d930>,
 'medical-0-S-0-R-3': <graphs.Node.Node at 0x1faa339d960>,
 'medical-0-S-0-R-4': <graphs.Node.Node at 0x1faa339d990>,
 'medical-0-S-1': <graphs.Node.Node at 0x1faa339d9c0>,
 'medical-0-S-1-R-0': <graphs.Node.Node at 0x1faa339dde0>,
 'medical-0-S-1-R-1': <graphs.Node.Node at 0x1faa339de10>,
 'medical-0-S-1-R-2': <graphs.Node.Node at 0x1faa339de40>,
 'medical-0-S-1-R-3': <graphs.Node.Node at 0x1faa339de70>,
 'medical-0-S-1-R-4': <graphs.Node.Node at 0x1faa339dea0>,
 'medical-0-S-1-R-5': <graphs.Node.Node at 0x1faa339ded0>,
 'medical-0-S-1-R-6': <graphs.Node.Node at 0x1faa339df00>,
 'medical-0-S-1-R-7': <graphs.Node.Node at 0x1faa339df60>,
 'medical-0-S-1-R-8': <graphs.Node.Node at 0x1faa339df30>,
 'medical-0-S-1-R-9': <graphs.Node.Node at 0x1faa339df90>,
 'med

In [4]:
import heapq

def dijkstra_with_paths(graph, src):
    dist = {src: 0}
    if graph[src].node_type == "N" and isinstance(graph[src].source, set):
        for syn in graph[src].source:
            dist[syn] = 0
    prev = {}
    pq = [(value,key) for key,value in dist.items()]

    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        for v, w in graph[u].edges.items():
            nd = d + 1/w
            if v not in dist or nd < dist[v]:
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))

    return dist, prev


In [5]:
def reconstruct_path(prev, src, dst):
    path = []
    cur = dst

    while cur != src:
        if cur not in prev:
            return None  # no path
        path.append(cur)
        cur = prev[cur]

    path.append(src)
    path.reverse()
    return path


In [6]:
from tqdm import tqdm
def all_pairs_shortest_paths(graph, entry_ids = None):
    all_paths = {}

    for src in tqdm(graph.keys()):
        if entry_ids and src not in entry_ids:
            continue 
        _, prev = dijkstra_with_paths(graph, src)
        all_paths[src] = {}

        for dst in graph:
            if src == dst:
                all_paths[src][dst] = [src]
            else:
                all_paths[src][dst] = reconstruct_path(prev, src, dst)

    return all_paths


In [7]:
import random
entry_node_id_list = list(medical_g6.keys())
entry_node_id_list = [i for i in entry_node_id_list if medical_g6[i].node_type in ["N", "O"]]
entry_nodes = random.sample(entry_node_id_list, 50)

In [8]:
paths = all_pairs_shortest_paths(medical_g6, entry_ids= entry_nodes)

100%|██████████| 43233/43233 [00:17<00:00, 2484.25it/s] 


In [42]:
l,x1,x2 = 0, None, None
for i in entry_nodes:
    for j in entry_nodes:
        p = paths[i][j]
        if p and len(p) >= l:
            x1 = i
            x2 = j
            l = len(p)
print("Max dist:", l)
print(x1,x2)
print(paths[x1][x2])

Max dist: 9
medical-N-2789 medical-N-8515
['medical-N-2789', 'medical-205-S-4', 'medical-205', 'medical-206', 'medical-207', 'medical-207-S-2', 'medical-N-2811', 'medical-517-S-4', 'medical-N-8515']


In [ ]:
for node_id in paths[x1][x2]:
    node = medical_g6[node_id]
    print(f"Node ID: {node_id}")
    print(f"Node Type: {node.node_type}")
    print(f"Node Source: {node.source}")
    print(f"Node Text: {node.content}")
    print("-----")

Node ID: medical-N-2789
Node Type: N
Node Source: {'medical-N-2787', 'medical-N-2785', 'medical-N-2788'}
Node Text: SPINAL TAP
-----
Node ID: medical-205-S-4
Node Type: S
Node Source: medical-205
Node Text: Leukemia can spread to the fluid around the spine or brain, leading to symptoms like headaches, neck pain, and light sensitivity. To determine if leukemia cells are present in the spinal fluid, a test called a lumbar puncture (LP) or spinal tap is performed. This procedure involves removing spinal fluid by inserting a needle into the middle of the lower back.
-----
Node ID: medical-205
Node Type: T
Node Source: 
Node Text: Certain treatments can cause a prolonged QTc. If the QTc becomes too prolonged, it can cause dangerous heart rhythms. Echocardiogram An echocardiogram (or echo) uses sound waves to make pictures. It is a type of ultrasound. For this test, small patches will be placed on your chest to track your heartbeat. Next, a wand with gel on its tip will be slid across part o

In [61]:
a = random.choice(entry_nodes)
b = random.choice(entry_nodes)
print("Path from", a, "to", b, "is:")
if not paths[a][b]:
    print("No path found")
else:
    print(len(paths[a][b]), "nodes long")
    print(paths[a][b])
    for node_id in paths[a][b]:
        node = medical_g6[node_id]
        print(f"Node ID: {node_id}")
        print(f"Node Type: {node.node_type}")
        print(f"Node Source: {node.source}")
        print(f"Node Text: {node.content}")
        print("-----")

Path from medical-N-2894 to medical-N-4273 is:
5 nodes long
['medical-N-2894', 'medical-527-S-0', 'medical-N-140', 'medical-223-S-1', 'medical-N-4273']
Node ID: medical-N-2894
Node Type: N
Node Source: {'medical-N-2888'}
Node Text: FALLOPIAN TUBES
-----
Node ID: medical-527-S-0
Node Type: S
Node Source: medical-527
Node Text: Chemotherapy can lead to infertility by damaging sperm or eggs, making fertility preservation an option for individuals of childbearing age who wish to have children after cancer treatment. Doctors will discuss fertility-related risks and may refer patients for counseling on options like semen cryopreservation, egg freezing (oocyte cryopreservation), ovarian tissue banking, and ovarian transposition (oophoropexy), which involves moving ovaries and fallopian tubes out of a radiation beam's range. Additional information is available in the NCCN Guidelines for Patients: Adolescents and Young Adults with Cancer.
-----
Node ID: medical-N-140
Node Type: N
Node Source: {